Loads the LoRA adapter from Drive and marks one example. Prompt format here must match training: system examiner prompt + `Question` / `Max Marks` / `Student Answer`.

In [ ]:
!pip install -q transformers==4.44.2 peft==0.13.0 accelerate==1.0.1 bitsandbytes==0.44.1


In [ ]:
# ==========================================
# FINAL TESTING BLOCK (COLAB READY)
# ==========================================

from google.colab import drive, userdata
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

# 1. Mount Google Drive to access your saved model
drive.mount('/content/drive')

# 2. Login to Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# 3. Setup Paths & Memory Optimization (The Fix)
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# POINT TO YOUR DRIVE FOLDER
# (Make sure this matches exactly where you saved it)
ADAPTER_PATH = "/content/drive/My Drive/feedbio_final_model"

# 4-Bit Config (Prevents RAM Crash)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading base model: {BASE_MODEL}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config, # <--- This saves 10GB of VRAM
    device_map="auto",
    offload_folder="./offload"
)

print(f"Loading adapter from: {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

# Step 4: Define feedback function
def generate_feedback(messages, max_new_tokens=500):
    # Step 1: Get the formatted string (e.g. "[INST] Question... [/INST]")
    # We use tokenize=False to just get the text first.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Step 2: Convert to Tensor (The safe way)
    # add_special_tokens=False prevents adding double "<s>" tokens
    model_inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    # Step 3: Generate
    print("Generating feedback...")
    output = model.generate(
        **model_inputs,  # This passes input_ids AND attention_mask automatically
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )

    # Step 4: Decode
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print("\n--- MODEL OUTPUT ---")
    print(decoded)

# Re-run the test
generate_feedback([
    {"role": "system", "content": "You are a strict but helpful GCSE Biology examiner. Your job is to mark answers and provide constructive feedback."},
    {"role": "user", "content": "Question: Explain why muscle cells have more mitochondria than skin cells.\nMax Marks: 3\nStudent Answer: Because they are big."}
])